# Chavruta.AI — Stage 1: fetch **Wikisource** sources (Kaggle · CPU)

מקביל ל-`fetch_licensed_kaggle.ipynb` (שמקורו Sefaria), אבל המקור כאן הוא **ויקיטקסט העברי**
(`he.wikisource.org`) — אתר שכל תוכנו מפורסם תחת **CC BY-SA 4.0 + GFDL באופן גורף**, לא per-חיבור
כמו ספריא. אין צורך לבדוק רישיון לכל מקור בנפרד — כל מה שנשלף מכאן CC-BY-SA, נקודה.

בונה **שתי שכבות חדשות** שאין לנו (או שאין לנו את המהדורה המסחרית שלהן) בקורפוס הקיים:

| slug | מקור | מה חסר לנו | ref/עיגון |
|---|---|---|---|
| `wikisource_kook` | 4 חיבורי הרב קוק (עין איה, שמונה קבצים, אורות התשובה, אגרות הראי"ה) | ספריא: כל הגרסאות `unknown`/לא קיים | `SOURCE` עצמאי |
| `wikisource_halacha` | משנה ברורה + בן איש חי | ספריא: `unknown` / לא קיים | משנה ברורה = `COMMENTARY` מעוגן לשו"ע הקיים; בן איש חי = `SOURCE` |

**חשוב — נבדק חי מול ה-API (2026-08-14), לא לפי הנחה. כולל תיקון באג אמיתי שנתפס בבדיקה:**

⚠️ **הבאג המרכזי שהמחברת הזאת נכתבה מחדש כדי לתקן:** הגרסה הראשונה השתמשה ב-
`prop=extracts&explaintext=1` (ה-API הסטנדרטי לטקסט נקי של MediaWiki). בדיקה על `אורות התשובה א`
גילתה שה-API הזה **משמיט בשקט טקסט שעטוף ב-`<div style="...">`** — וזה בדיוק איך שדפי הרב קוק
מעוצבים. התוצאה שהתקבלה בפועל: **הטקסט המקורי של הרב קוק נעלם, ומה שנשאר הוא רק "סיכום" — פרשנות
פדגוגית מודרנית שכתב עורך ויקיטקסט אנונימי.** זה בדיוק המצב שהחוקה אוסרת: ציטוט של פרפראזה בתור
מקור ראשוני. **המחברת הזו שולפת `wikitext` גולמי (`prop=revisions`) ומנקה אותו בעצמה**, לא סומכת
על חילוץ הטקסט של MediaWiki. נבדק חוזר על אורות התשובה, עין איה, משנה ברורה ובן איש חי — כולם
מחזירים כעת את הטקסט המקורי המלא, לא רק תקציר.

בנוסף מסננת **החוצה** לפי כותרת סעיף כל מה שמזוהה כתוספת עריכתית מודרנית (`סיכום`, `ביאור עורך`
וכו') — גם כשהיא לצד הטקסט המקורי באותו עמוד.

**מבנה שנבדק:**
- עין איה: 1,956 עמודים בקטגוריה, כולל עמודי ניווט שצריך לסנן (רק `עין איה על ...`).
- שמונה קבצים: 9 עמודים (8 קבצים + אינדקס).
- אורות התשובה: 5 עמודים.
- אגרות הראי"ה: **קטן ולא שלם** — נמצאו רק 2 עמודים ישירים, כנראה רק אגרות שלא נכללו במהדורה
  הרשמית של מוסד הרב קוק. מטופל כ**best-effort**, לא כמקור שלם — אל תסמכו על שלמותו.
- משנה ברורה: קטגוריית "משנה ברורה על אורח חיים" (לא ביאור הלכה — קטגוריה נפרדת), מחולקת לפי
  `=== סעיף X ===`.
- בן איש חי: קטגוריית "הלכות" (110 עמודים), מבנה שנה→פרשה→הלכה, מחולק לפי `== פתיחה ==`/`== <הלכה> ==`.

`unit_type=COMMENTARY` דורש `anchor_ref` + `commentator_id` ([schema.py](../src/chavruta/corpus/schema.py)
— `validate()`); משנה ברורה מקבל את שניהם. שאר המקורות נכנסים כ-`SOURCE` בלי עיגון — בדיוק כמו
שכבר טעונים חיבורי הרב קוק הקיימים (Orot וכו').

⚠️ **לפני שמריצים בפרודקשן:** ה-`anchor_ref` שמחברת זו בונה למשנה ברורה מבוסס על הכלל התיעודי
ב-`corpus/refs.py` (צורה מקווקוות בסגנון ספריא) — **לא אומת מול הקולקציה החיה** כי לא הייתה גישה
אליה מסשן המחקר. יש תא בדיקה (סעיף 9) שמדפיס דוגמאות `anchor_ref` — תריצו אותו ותשוו ל-`ref`
האמיתי של Shulchan Arukh בקולקציה שלכם לפני שסומכים על העיגון.

⚠️ **הניקוי (regex-based, לא פרסר wikitext מלא) מכסה את התבניות שנצפו בפועל בארבע הדוגמאות
שנבדקו** (`{{סרגל ניווט}}`, `{{ב|...}}`, `{{קטן|...}}`, `{{צתב|...}}`, `{{משע|...}}`, `<div>`,
`'''`/`''`, `[[...]]`). ייתכנו תבניות אחרות בעמודים שלא נבדקו — תא הביקורת (סעיף 11) מזהיר אם
נשארו סוגריים מסולסלים/מרובעים בטקסט שנוקה, כדי לתפוס עוד מקרים לפני העלאה, לא אחריה.

⚠️ **תיקון נוסף (2026-08-21): `deep_link`.** גילינו ש-`corpus/ingest.py::payload_from_legacy_meta`
בונה תמיד `https://www.sefaria.org/{ref}` — קישור מת לכל תוכן שאינו מספריא. המחברת הזו עכשיו
פולטת `deep_link` אמיתי לעמוד ויקיטקסט המקורי (`wikisource_page_url()`, סעיף 8), ו-`ingest.py`
תוקן להעדיף אותו כשהוא קיים. נבדק: `Wikisource_wikisource_kook:אורות התשובה א#0.0` -> `https://
he.wikisource.org/wiki/%D7%90%D7%95%D7%A8%D7%95%D7%AA_...` — קישור אמיתי לעמוד הנכון, לא לספריא.

## 1. Config — התא שעורכים

In [ ]:
# שני מאגרי HF חדשים — לא נוגעים בקיימים
HF_NAMESPACE = "Yehuda-Rubin"
REPO_KOOK    = f"{HF_NAMESPACE}/chavruta-wikisource-kook"
REPO_HALACHA = f"{HF_NAMESPACE}/chavruta-wikisource-halacha"

# אילו שכבות להריץ עכשיו (שתיהן קטנות — אין צורך לפצל לכמה הרצות, בניגוד ל-fetch_licensed_kaggle)
TIERS_TO_RUN = ["wikisource_kook", "wikisource_halacha"]

USER_AGENT = "Chavruta.AI/0.1 (educational Torah RAG; contact: rubinri@gmail.com)"


## 2. התקנות + כניסה ל-HF

In [ ]:
!pip install -q "requests>=2.32" "huggingface_hub>=0.23"

**הרצה ידנית בכל פעם** — תיבת קלט בתא הבא, לא Kaggle Secrets ולא משתנה סביבה.
טוקן עם הרשאת **Write**: https://huggingface.co/settings/tokens

In [ ]:
import getpass
from huggingface_hub import login, HfApi

tok = getpass.getpass("HF write token: ")
login(token=tok); HF = HfApi()
print("logged in as:", HF.whoami()["name"])


## 3. Wikisource API — enumerate + fetch **raw wikitext** (לא plaintext extracts)

`action=query&list=categorymembers` לרשימת עמודים. `action=query&prop=revisions&rvprop=content`
לתוכן הגולמי — **בקשה אחת לעמוד**, לא batched: נבדק חי (2026-08-14) ש-`prop=extracts` עם כמה
`titles=` יחד מחזיר תוכן רק לכותרת הראשונה (גם עם `exlimit=max`).

In [ ]:
import time
import requests

WS_API = "https://he.wikisource.org/w/api.php"
S = requests.Session()
S.headers.update({"User-Agent": USER_AGENT})


def _get(params, retries=4):
    params = {**params, "format": "json"}
    for a in range(retries):
        try:
            r = S.get(WS_API, params=params, timeout=60)
        except requests.RequestException:
            time.sleep(2 * (a + 1)); continue
        if r.status_code == 200:
            try:
                return r.json()
            except Exception:
                return {}
        time.sleep(2 * (a + 1))
    return {}


def category_pages(category, ns=0):
    """כל דפי התוכן (ns=0) בקטגוריה נתונה, עם דפדוף מלא (cmcontinue)."""
    cont, titles = None, []
    while True:
        params = {"action": "query", "list": "categorymembers", "cmtitle": category,
                  "cmlimit": "500", "cmnamespace": str(ns)}
        if cont:
            params["cmcontinue"] = cont
        d = _get(params)
        members = d.get("query", {}).get("categorymembers", [])
        titles.extend(m["title"] for m in members)
        cont = d.get("continue", {}).get("cmcontinue")
        if not cont:
            break
    return titles


def fetch_wikitext(titles):
    """title -> raw wikitext, ONE title per request (ראו האזהרה למעלה)."""
    out = {}
    for t in titles:
        d = _get({"action": "query", "prop": "revisions", "rvprop": "content",
                  "rvslots": "main", "redirects": "1", "titles": t})
        for _, p in d.get("query", {}).get("pages", {}).items():
            revs = p.get("revisions") or [{}]
            out[t] = revs[0].get("slots", {}).get("main", {}).get("*", "") or ""
        time.sleep(0.15)
    return out


## 4. ניקוי wikitext → טקסט נקי

**נבנה ונבדק חי** מול 4 עמודים אמיתיים (ראו סעיף 0). לא פרסר wikitext כללי — regex + מעקב-עומק
ל-`{{תבניות מקוננות}}`, שמכסה את מה שבפועל נמצא בעמודים האלה. `_LAST_PARAM_TEMPLATES` הוא רשימת
תבניות שבהן התוכן האמיתי הוא הפרמטר **האחרון** (כמו `{{משע|תווית|אות סעיף|טקסט}}` במשנה ברורה,
לעומת `{{ב|תצוגה|הסבר}}` שבו התוכן הוא הפרמטר הראשון) — נמצא רק אחרי שתא הביקורת בסעיף 11 תפס
שריד "מב" בתחילת פסקאות.

In [ ]:
import re

_NAV_TEMPLATES = {"סרגל ניווט", "עוגן", "הבהרה", "ניווט"}          # תבניות ניווט — מוסרות לגמרי
_LAST_PARAM_TEMPLATES = {"משע"}    # {{משע|תווית|אות סעיף|טקסט הלמה}} -> התוכן הוא הפרמטר האחרון


def strip_templates(text):
    """מסיר {{תבניות}}, כולל מקוננות: התוכן הפנימי מעובד רקורסיבית לפני הפיצול לפרמטרים —
    אחרת תבנית מקוננת דולפת כטקסט גולמי."""
    out = []
    i, n = 0, len(text)
    while i < n:
        if text[i:i + 2] == "{{":
            depth = 1
            j = i + 2
            while j < n and depth > 0:
                if text[j:j + 2] == "{{":
                    depth += 1; j += 2
                elif text[j:j + 2] == "}}":
                    depth -= 1; j += 2
                else:
                    j += 1
            inner = strip_templates(text[i + 2:j - 2])   # תבניות מקוננות נפתרות קודם
            parts = inner.split("|")
            name = parts[0].strip()
            if name in _NAV_TEMPLATES:
                pass
            elif name in _LAST_PARAM_TEMPLATES and len(parts) > 1:
                out.append(parts[-1])
            elif len(parts) > 1:
                out.append(parts[1])
            i = j
        else:
            out.append(text[i]); i += 1
    return "".join(out)


def wikitext_to_text(wt):
    t = strip_templates(wt)
    t = re.sub(r"<[^>]+>", "", t)                            # תגי HTML — התוכן נשמר, רק התג יורד
    t = re.sub(r"\[\[קטגוריה:[^\]]*\]\]", "", t)               # קישורי קטגוריה — לא תוכן
    t = re.sub(r"\[\[[^\]|]*\|([^\]]*)\]\]", r"\1", t)         # [[יעד|תצוגה]] -> תצוגה
    t = re.sub(r"\[\[([^\]]*)\]\]", r"\1", t)                   # [[יעד]] -> יעד
    t = t.replace("\'\'\'", "").replace("\'\'", "")               # הדגשה/נטוי
    t = re.sub(r"^[:*#]+\s*", "", t, flags=re.MULTILINE)       # תחיליות רשימה/הזחה
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


## 5. גימטריה — המרת אות עברית למספר (לעיגון משנה ברורה לשו"ע)

In [ ]:
_GEMATRIA = {
    "א": 1, "ב": 2, "ג": 3, "ד": 4, "ה": 5, "ו": 6, "ז": 7, "ח": 8, "ט": 9,
    "י": 10, "כ": 20, "ל": 30, "מ": 40, "נ": 50, "ס": 60, "ע": 70, "פ": 80, "צ": 90,
    "ק": 100, "ר": 200, "ש": 300, "ת": 400,
}


def gematria_to_int(s):
    """'תמח' -> 448. מתעלם מגרש/גרשיים, כי לצורך סכימה הם לא נושאים ערך."""
    total = 0
    for ch in s:
        if ch in _GEMATRIA:
            total += _GEMATRIA[ch]
    return total


assert gematria_to_int("א") == 1
assert gematria_to_int("תמח") == 448          # משנה ברורה על אורח חיים תמח
assert gematria_to_int("שצו") == 396
print("gematria_to_int OK")


## 6. חיתוך לצ'אנקים — לפי כותרות `==`/`===`, וסינון תוספת עריכתית

**כותרות שמזוהות כתוספת עריכתית מודרנית (לא הטקסט המקורי) נדחות כאן, לא רק בשלב הבנייה** —
`סיכום` הוא הדוגמה שנמצאה בפועל באורות התשובה; הרשימה מוצהרת ולא ממצה, תוסיפו אליה אם תתגלה עוד.

In [ ]:
_SECTION_RE = re.compile(r"^={2,3}\s*(.+?)\s*={2,3}\s*$", re.MULTILINE)
_EDITORIAL_HEADINGS = {"סיכום", "ביאור עורך", "הערת עורך", "שאלות לחזרה", "לסיכום"}
MIN_CHARS, MAX_CHARS = 400, 2000


def split_sections(text):
    """מפצל טקסט לפי כותרות == כותרת == / === כותרת ===.
    מחזיר [(heading_or_None, body), ...], בלי כותרות שזוהו כעריכה מודרנית ולא כטקסט מקור."""
    matches = list(_SECTION_RE.finditer(text))
    if not matches:
        return [(None, text.strip())] if text.strip() else []
    out = []
    pre = text[:matches[0].start()].strip()          # טקסט לפני הכותרת הראשונה (הקדמה/ניווט)
    if pre:
        out.append((None, pre))
    for i, m in enumerate(matches):
        heading = m.group(1).strip()
        if heading in _EDITORIAL_HEADINGS:
            continue                                   # תוספת עריכתית — לא טקסט המקור, לא נכנס
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            out.append((heading, body))
    return out


def chunk_body(body, min_chars=MIN_CHARS, max_chars=MAX_CHARS):
    """פסקה ארוכה מדי מתפצלת בגבול פסקה (\n\n) הכי קרוב לאמצע."""
    if len(body) <= max_chars:
        return [body]
    paras = [p for p in body.split("\n\n") if p.strip()]
    out, buf = [], ""
    for p in paras:
        if buf and len(buf) + len(p) > max_chars:
            out.append(buf.strip()); buf = p
        else:
            buf = f"{buf}\n\n{p}" if buf else p
    if buf.strip():
        out.append(buf.strip())
    return out or [body]


## 7. הגדרת המקורות — קטגוריה/דפים לכל חיבור

עיגון (`anchor_ref`) הוא **אך ורק** למשנה ברורה — נבנה ב-`mishnah_berura_anchor()` למטה ונקרא
ישירות מ-`build_chunks` לפי שם הספר, לא דרך שדה per-מקור. שאר החיבורים הם `SOURCE` בלי עיגון,
בדיוק כמו שכבר טעון עבור חיבורי הרב קוק הקיימים.

In [ ]:
def mishnah_berura_anchor(page_title):
    """'משנה ברורה על אורח חיים תמח' -> anchor_ref בסגנון ספריא (ראה אזהרה בסעיף 0 —
    לא אומת מול הקולקציה החיה). corpus/refs.py::with_ref_variants מפיק את שני האיות
    (מקווקו/עם רווח), כך שצורה זו אמורה להתיישר עם מה שכבר טעון."""
    m = re.match(r"^משנה ברורה על אורח חיים (.+)$", page_title.strip())
    if not m:
        return None
    n = gematria_to_int(m.group(1))
    if not n:
        return None
    return f"Shulchan Arukh, Orach Chayim.{n}"


SOURCES = {
    "wikisource_kook": [
        {
            "work_id": "wikisource_kook", "author_he": "הרב אברהם יצחק הכהן קוק",
            "book": "Ein Ayah", "he_book": "עין איה",
            "category": "קטגוריה:עין איה",
            "keep": lambda t: t.startswith("עין איה על "),
            "commentator_id": None, "period": "modern",
        },
        {
            "work_id": "wikisource_kook", "author_he": "הרב אברהם יצחק הכהן קוק",
            "book": "Shemonah Kevatzim", "he_book": "שמונה קבצים",
            "category": "קטגוריה:שמונה קבצים",
            "keep": lambda t: t != "שמונה קבצים",       # הדף הראשי הוא ניווט, לא תוכן
            "commentator_id": None, "period": "modern",
        },
        {
            "work_id": "wikisource_kook", "author_he": "הרב אברהם יצחק הכהן קוק",
            "book": "Orot HaTeshuvah", "he_book": "אורות התשובה",
            "category": "קטגוריה:אורות התשובה",
            "keep": lambda t: t != "אורות התשובה",
            "commentator_id": None, "period": "modern",
        },
        {
            # best-effort — ראה אזהרה בסעיף 0: לא קטגוריה שלמה, רשימת דפים ידנית קטנה.
            "work_id": "wikisource_kook", "author_he": "הרב אברהם יצחק הכהן קוק",
            "book": "Igrot HaRe'iyah", "he_book": "אגרות הראי\"ה",
            "pages": ["אגרות הראי\"ה", "אגרות הראיה ה"],
            "commentator_id": None, "period": "modern",
        },
    ],
    "wikisource_halacha": [
        {
            "work_id": "wikisource_halacha", "author_he": "החפץ חיים (ישראל מאיר הכהן)",
            "book": "Mishnah Berurah", "he_book": "משנה ברורה",
            "category": "קטגוריה:משנה ברורה על אורח חיים",
            "keep": lambda t: True,
            "commentator_id": "mishnah_berurah", "period": "acharonim",
        },
        {
            "work_id": "wikisource_halacha", "author_he": "רבי יוסף חיים מבגדד",
            "book": "Ben Ish Chai", "he_book": "בן איש חי",
            "category": "קטגוריה:בן איש חי הלכות",
            "keep": lambda t: True,
            "commentator_id": None, "period": "acharonim",
        },
    ],
}


## 8. Chunk builder — עמוד ויקיטקסט → רשומות בפורמט הקיים

אותה סכמת שדות בדיוק כמו שאר שכבות הקורפוס (`id`/`document`/`metadata`), כדי שהמחברת
של Lightning (שלב הבא) תוכל להשתמש באותו קוד הטמעה בלי שינוי.

In [ ]:
import urllib.parse
from datetime import datetime, timezone

LICENSE = "CC-BY-SA"                       # קבוע — כל תוכן ויקיטקסט, לא per-מקור
VERSION_PREFIX = "he.wikisource.org"


def wikisource_page_url(page_title):
    """כתובת עמוד אמיתית — נבדק חי מול URL אמיתי (רווח -> קו תחתון, ואז quote).
    בלי זה, ingest.py::payload_from_legacy_meta היה בונה sefaria.org/<ref-של-ויקיטקסט> —
    קישור מת. עכשיו deep_link מפורש במטא-דאטה, וה-loader מעדיף אותו (תוקן 2026-08-21)."""
    return "https://he.wikisource.org/wiki/" + urllib.parse.quote(page_title.replace(" ", "_"))


def build_chunks(source, page_titles, wikitexts):
    out, n_pages_used = [], 0
    is_mb = source["book"] == "Mishnah Berurah"
    for page_title in page_titles:
        wt = wikitexts.get(page_title, "")
        if not wt.strip():
            continue
        text = wikitext_to_text(wt)
        if not text.strip():
            continue
        n_pages_used += 1
        sections = split_sections(text)
        if not sections:
            continue
        for sec_i, (heading, body) in enumerate(sections):
            for part_i, part in enumerate(chunk_body(body)):
                if len(part) < 40:              # רעש/כותרת יתומה — לא צ'אנק בפני עצמו
                    continue
                ref = f"Wikisource_{source['work_id']}:{page_title}#{sec_i}.{part_i}"
                anchor_ref = mishnah_berura_anchor(page_title) if is_mb else None
                rec = {
                    "id": re.sub(r"[^\w]+", "_", ref, flags=re.UNICODE).strip("_"),
                    "document": part,
                    "metadata": {
                        "verse_id": ref,
                        "ref": ref,
                        "book": source["book"],
                        "chunk_type": source["work_id"],
                        "commentator": source.get("commentator_id") or "",
                        "work": source["work_id"],
                        "period": source["period"],
                        "author_he": source["author_he"],
                        "section": heading or page_title,
                        "text_he": part,
                        "text_en": "",
                        "license_he": LICENSE,
                        "version_he": f"{VERSION_PREFIX} \u2014 {page_title}",
                        "license_en": "",
                        "version_en": "",
                        "deep_link": wikisource_page_url(page_title),
                        "anchor_ref": anchor_ref,
                        "anchor_kind": "source" if anchor_ref else None,
                        "unit_type": "commentary" if anchor_ref else "source",
                    },
                }
                out.append(rec)
    return out, n_pages_used


## 9. Tier runner — שולף כל מקור בשכבה, כותב JSONL, בונה מניפסט

In [ ]:
import json
from pathlib import Path

WORK = Path("work"); WORK.mkdir(exist_ok=True)


def run_source(source):
    if "pages" in source:
        titles = source["pages"]
    else:
        titles = [t for t in category_pages(source["category"]) if source["keep"](t)]
    print(f"  - {source['book']}: {len(titles)} candidate pages", flush=True)
    wikitexts = fetch_wikitext(titles)
    chunks, n_used = build_chunks(source, titles, wikitexts)
    print(f"    -> {n_used}/{len(titles)} pages had text, {len(chunks)} chunks", flush=True)
    return chunks


def build_tier(slug):
    out_path = WORK / f"{slug}.jsonl"
    manifest = []
    with out_path.open("w", encoding="utf-8") as f:
        total = 0
        for source in SOURCES[slug]:
            chunks = run_source(source)
            for c in chunks:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")
            total += len(chunks)
            manifest.append({
                "book": source["book"], "he_book": source["he_book"],
                "author_he": source["author_he"], "chunks": len(chunks),
                "license": LICENSE, "status": "kept" if chunks else "no_content_found",
            })
    print(f"[{slug}] {total:,} chunks total -> {out_path}", flush=True)
    return out_path, total, manifest


## 10. הרצה

In [ ]:
BUILT, MANIFESTS = {}, {}
for slug in TIERS_TO_RUN:
    print(f"\n=== {slug} ===", flush=True)
    path, n, manifest = build_tier(slug)
    MANIFESTS[slug] = manifest
    if n:
        BUILT[slug] = path
print("\nbuilt:", {k: str(v) for k, v in BUILT.items()})


## 11. ביקורת לפני העלאה — כולל בדיקת דליפת סימוני wikitext

מזהירה אם נשארו `{{`/`}}`/`[[`/`]]` בטקסט אחרי הניקוי — סימן לתבנית שלא טופלה נכון (ראו האזהרה
בסעיף 4). לא עוצרת אוטומטית, כי זה עלול להיות false positive על תוכן לגיטימי, אבל **תבדקו ידנית**
כל דוגמה שמודפסת כאן לפני שמעלים.

In [ ]:
for slug, manifest in MANIFESTS.items():
    print(f"--- {slug} ---")
    for m in manifest:
        flag = "OK " if m["status"] == "kept" else "?? "
        print(f"  {flag}{m['he_book']:20s} {m['chunks']:6,} chunks  ({m['status']})")

assert all(any(m["status"] == "kept" for m in MANIFESTS[s]) for s in MANIFESTS), \
    "תריצה שלא הפיקה שום צ'אנק — לבדוק לפני העלאה"

print("\nחיפוש דליפת wikitext ('{{', '}}', '[[', ']]') בצ'אנקים שנבנו:")
leaks = 0
for slug, path in BUILT.items():
    with open(path, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            t = d["document"]
            if any(m in t for m in ("{{", "}}", "[[", "]]")):
                leaks += 1
                if leaks <= 5:
                    print(f"  ?? {d['metadata']['ref']}: {t[:120]!r}")
print(f"total possible leaks: {leaks}" + (" -- check manually before uploading!" if leaks else " -- clean"))


## 12. בדיקת עיגון — ⚠️ תריצו לפני שסומכים על `anchor_ref` של משנה ברורה

מדפיס כמה refs שנבנו, כדי שתשוו ידנית ל-`ref` האמיתי של Shulchan Arukh בקולקציה החיה שלכם.

In [ ]:
sample = []
mb_path = BUILT.get("wikisource_halacha")
if mb_path:
    with open(mb_path, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            if d["metadata"].get("anchor_ref"):
                sample.append((d["metadata"]["book"], d["metadata"]["ref"], d["metadata"]["anchor_ref"]))
            if len(sample) >= 8:
                break
for book, ref, anchor in sample:
    print(f"{book:16s} ref={ref!r:60s} anchor_ref={anchor!r}")
if not sample:
    print("אין דוגמאות עיגון (משנה ברורה לא נכלל בהרצה הזו)")


## 13. README + `licenses.json` לכל repo

אותה מוסכמה כמו `fetch_licensed_kaggle.ipynb` — קובץ מניפסט קריא-למכונה + כרטיס dataset.

In [ ]:
REPO_DOCS = {}

def write_repo_docs(slug, manifest):
    lic_json = WORK / f"{slug}.licenses.json"
    lic_json.write_text(json.dumps({
        "tier": slug, "source": "he.wikisource.org",
        "license": LICENSE, "license_url": "https://creativecommons.org/licenses/by-sa/4.0/",
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "sources": manifest,
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    total = sum(m["chunks"] for m in manifest)
    lines = [
        f"# {slug}\n",
        "\nChavruta.AI -- Wikisource-sourced tier. Site-wide **CC BY-SA 4.0 + GFDL**",
        f" (he.wikisource.org). {total:,} chunks from {len(manifest)} works.\n",
        "\nNot legal advice. Attribution + share-alike required on reproduction (see repo docs).\n",
        "\n## Sources\n",
    ]
    for m in manifest:
        lines.append(f"- **{m['he_book']}** ({m['author_he']}) -- {m['chunks']:,} chunks -- {m['status']}\n")
    readme = WORK / f"{slug}.README.md"
    readme.write_text("".join(lines), encoding="utf-8")
    return str(lic_json), str(readme)

REPO_BY_SLUG = {"wikisource_kook": REPO_KOOK, "wikisource_halacha": REPO_HALACHA}
for slug, manifest in MANIFESTS.items():
    REPO_DOCS[slug] = write_repo_docs(slug, manifest)
print("docs written:", list(REPO_DOCS.keys()))


## 14. העלאה ל-HF — שני repos חדשים

In [ ]:
from huggingface_hub import create_repo

for slug, path in BUILT.items():
    repo = REPO_BY_SLUG[slug]
    create_repo(repo, repo_type="dataset", exist_ok=True, token=tok)
    lic_json, readme = REPO_DOCS[slug]
    print(f"uploading {slug} -> {repo}", flush=True)
    HF.upload_file(path_or_fileobj=str(path),  path_in_repo=f"{slug}.jsonl", repo_id=repo, repo_type="dataset", token=tok)
    HF.upload_file(path_or_fileobj=lic_json,   path_in_repo="licenses.json", repo_id=repo, repo_type="dataset", token=tok)
    HF.upload_file(path_or_fileobj=readme,     path_in_repo="README.md",     repo_id=repo, repo_type="dataset", token=tok)
    print(f"   https://huggingface.co/datasets/{repo}")
print("\ndone.")


## הבא — שלב 2 (GPU, Lightning)

`embed_extend_lightning.ipynb` מוריד את שני ה-repos האלה **+ את האינדקס המפורסם הקיים**
(`chavruta-commercial-index`), מטמיע רק את הצ'אנקים החדשים (bge-m3), מוסיף אותם על גבי
הקולקציה הקיימת (לא נוגע ב-2.4M הנקודות הקיימות), ומפרסם אינדקס חדש נפרד — לא דורס פרודקשן.